In [66]:
import pandas as pd
import numpy as np
import os
import glob
import re
from tqdm import tqdm



In [67]:

from pathlib import Path

# Get current working directory as a Path object
current_dir = Path.cwd()

# Get parent directory
root = current_dir.parent

print(f"Root directory set to: {root}")

FOLDER = root / "outputs" / "csv"/"wifi-random"
OUTPUT = root / "outputs" / "combine_analysis" / "final_merged_training_dataset.csv"

FILE_REGEX = r"wifi-random-(.*?)-seed(\d+)-(.+?)\.csv"

Root directory set to: e:\AIT_Projects\NS3-Project


In [68]:

def extract_run_id(filename):
    m = re.search(FILE_REGEX, filename)
    if not m:
        return None
    return f"{m.group(1)}-seed{m.group(2)}"


# ============================================================
# Scan folder and group files by simulation
# ============================================================
def scan_runs(folder):
    runs = {}
    for f in glob.glob(os.path.join(folder, "*.csv")):
        base = os.path.basename(f)
        run_id = extract_run_id(base)
        if run_id is None:
            continue

        runs.setdefault(run_id, {})
        if "nodedensity" in base:
            runs[run_id]["nd"] = f
        elif "modulation" in base:
            runs[run_id]["mod"] = f
        elif "perf" in base:
            runs[run_id]["perf"] = f
    return runs


# ============================================================
# Build NodeDensity intervals
# ============================================================
def build_intervals(df_nd):
    df = df_nd.copy()
    df["t_start"] = df["Duration_s"].cumsum().shift(fill_value=0)
    df["t_end"] = df["t_start"] + df["Duration_s"]
    return df


# ============================================================
# Compute interval features from modulation rows
# Includes RSSI, Noise, SNR_computed, PhyRate
# PLUS automatic encoding of Modulation types + channel frequencies
# ============================================================
def compute_interval_mod_features(df_mod, t_start, t_end):
    df = df_mod.copy()

    df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")
    df = df.dropna(subset=["time_s"])
    df["SNR_computed"] = df["signal_dBm"] - df["noise_dBm"]

    mask = (df["time_s"] >= t_start) & (df["time_s"] <= t_end)
    df_slice = df.loc[mask]

    if df_slice.empty:
        return {}

    features = {
        # Base PHY stats
        "Mod_RSSI_mean": df_slice["signal_dBm"].mean(),
        "Mod_RSSI_min": df_slice["signal_dBm"].min(),
        "Mod_RSSI_max": df_slice["signal_dBm"].max(),

        "Mod_Noise_mean": df_slice["noise_dBm"].mean(),

        "Mod_SNR_mean": df_slice["SNR_computed"].mean(),

        "Mod_PhyRate_mean": df_slice["PhyRate_Mbps"].mean(),
        "Mod_PhyRate_max": df_slice["PhyRate_Mbps"].max(),

        "IntervalSamples": len(df_slice)
    }

    # Add Modulation counts
    modulation_counts = df_slice["Modulation"].value_counts().to_dict()
    for mod_type, count in modulation_counts.items():
        features[f"Modulation_{mod_type}_count"] = count

    # Add channel counts
    channel_counts = df_slice["channel_MHz"].value_counts().to_dict()
    for ch, count in channel_counts.items():
        features[f"Channel_{ch}_count"] = count

    return features


# ============================================================
# Aggregate perf.csv
# ============================================================
def compute_perf_features(df_perf):
    return {
        "Perf_Throughput": df_perf["Throughput(Mbps)"].mean(),
        "Perf_Latency": df_perf["Latency_avg(ms)"].mean(),
        "Perf_Jitter": df_perf["Jitter_avg(ms)"].mean(),
        "Perf_Loss": df_perf["PacketLoss(%)"].mean(),
    }


# ============================================================
# MAIN BUILD FUNCTION
# ============================================================
def build_dataset(runs):
    all_rows = []

    for run_id, fdict in tqdm(runs.items(), desc="Building dataset"):

        if "nd" not in fdict:
            continue

        df_nd = pd.read_csv(fdict["nd"])
        df_nd = build_intervals(df_nd)

        df_mod = pd.read_csv(fdict["mod"]) if "mod" in fdict else None

        df_perf = pd.read_csv(fdict["perf"]) if "perf" in fdict else None
        perf_stats = compute_perf_features(df_perf) if df_perf is not None else {}

        for _, ndrow in df_nd.iterrows():
            record = ndrow.to_dict()
            record["RunID"] = run_id

            if df_mod is not None:
                mod_stats = compute_interval_mod_features(
                    df_mod,
                    t_start=float(ndrow["t_start"]),
                    t_end=float(ndrow["t_end"])
                )
                record.update(mod_stats)

            record.update(perf_stats)

            all_rows.append(record)

    return pd.DataFrame(all_rows)


# ============================================================
# EXECUTE
# ============================================================
runs = scan_runs(FOLDER)
df_final = build_dataset(runs)
df_final.to_csv(OUTPUT, index=False)

df_final


Building dataset: 100%|██████████| 3/3 [00:06<00:00,  2.20s/it]


,StartDateTime,EndDateTime,Duration_s,NodeDensity,TotalTxPackets,TotalRxPackets,AvgThroughput(Mbps),PacketLoss(%),AvgLatency(ms),AvgJitter(ms),AvgRSSI(dBm),AvgSNR(dB),AvgBER,t_start,t_end,RunID,Perf_Throughput,Perf_Latency,Perf_Jitter,Perf_Loss
0,2025-12-09 16:22:46,2025-12-09 16:22:51,5,7,47600,3695,0.370688,92.2374,50.0787,8.05943,-77.1025,16.8634,4.277420e-308,0,5,09-Dec-2025_16-22-seed12345,0.268315,21.283428,3.425256,96.700895
1,2025-12-09 16:22:51,2025-12-09 16:22:56,5,10,28000,2831,0.203942,89.8893,27.0152,6.47121,0.0000,0.0000,0.000000e+00,5,10,09-Dec-2025_16-22-seed12345,0.268315,21.283428,3.425256,96.700895
2,2025-12-09 16:22:56,2025-12-09 16:23:01,5,11,25200,2461,0.156391,90.2341,24.8748,6.15728,0.0000,0.0000,0.000000e+00,10,15,09-Dec-2025_16-22-seed12345,0.268315,21.283428,3.425256,96.700895
3,2025-12-09 16:23:01,2025-12-09 16:23:11,10,9,19600,1491,0.088746,92.3929,17.6066,5.76820,0.0000,0.0000,0.000000e+00,15,25,09-Dec-2025_16-22-seed12345,0.268315,21.283428,3.425256,96.700895
4,2025-12-09 16:23:11,2025-12-09 16:23:16,5,11,8400,555,0.056160,93.3929,26.4712,8.41560,0.0000,0.0000,0.000000e+00,25,30,09-Dec-2025_16-22-seed12345,0.268315,21.283428,3.425256,96.700895
5,2025-12-09 16:23:16,2025-12-09 16:23:16,0,12,2800,290,0.087150,89.6429,60.7290,11.52960,0.0000,0.0000,0.000000e+00,30,30,09-Dec-2025_16-22-seed12345,0.268315,21.283428,3.425256,96.700895
6,2025-12-09 16:50:16,2025-12-09 16:50:21,5,7,47600,3695,0.370688,92.2374,50.0787,8.05943,-77.1025,16.8634,4.277420e-308,0,5,09-Dec-2025_16-49-seed12345,0.268315,21.283428,3.425256,96.700895
7,2025-12-09 16:50:21,2025-12-09 16:50:26,5,10,28000,2831,0.203942,89.8893,27.0152,6.47121,0.0000,0.0000,0.000000e+00,5,10,09-Dec-2025_16-49-seed12345,0.268315,21.283428,3.425256,96.700895
8,2025-12-09 16:50:26,2025-12-09 16:50:31,5,11,25200,2461,0.156391,90.2341,24.8748,6.15728,0.0000,0.0000,0.000000e+00,10,15,09-Dec-2025_16-49-seed12345,0.268315,21.283428,3.425256,96.700895
9,2025-12-09 16:50:31,2025-12-09 16:50:41,10,9,19600,1491,0.088746,92.3929,17.6066,5.76820,0.0000,0.0000,0.000000e+00,15,25,09-Dec-2025_16-49-seed12345,0.268315,21.283428,3.425256,96.700895
